# 31. Hybrid Feature Fusion + Cross-Encoder (분할 실행)

기존 29번의 통합 데이터·HS6 적격 기준·제품군 정의·LOPO 평가를 그대로 사용합니다. 셀을 위에서부터 순서대로 실행하세요. Google Drive에 중간 결과와 임베딩을 저장하므로 런타임이 끊겨도 완료된 Fold부터 이어갈 수 있습니다.

## 0. 환경 설정 및 Drive 연결

처음 한 번만 패키지를 설치하고 Drive를 연결합니다.

In [ ]:
!pip -q install -U sentence-transformers kiwipiepy scipy
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, requests, zipfile, re, numpy as np, pandas as pd
from sentence_transformers import CrossEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support
from sklearn.model_selection import LeaveOneGroupOut
WORK = Path('/content/drive/MyDrive/HS_complete_exp31')
WORK.mkdir(parents=True, exist_ok=True)
print('저장 위치:', WORK)

## 1. 29번 데이터 파이프라인 재현

기존 노트북의 데이터 정제·제품군·평가 대상 정의만 실행합니다. SBERT 임베딩은 다음 셀에서 별도로 생성합니다.

In [ ]:
BASE='https://raw.githubusercontent.com/lin1118-code/HS-complete-product/feature/explanatory-note-integration/'
src=requests.get(BASE+'29_discriminative_features_experiment.ipynb').json()
for i in [3,5,7]:
    exec(''.join(src['cells'][i]['source']), globals())
print('평가 데이터:',len(eval_df),'| HS6:',len(labels),'| 제품군:',eval_df.product_family.nunique())
assert len(eval_df)==195 and len(labels)==11 and eval_df.product_family.nunique()==29, '평가 모집단이 기존 기준과 다릅니다.'

## 2. SBERT 임베딩 생성 또는 캐시 불러오기

최초 실행에서 시간이 가장 오래 걸릴 수 있습니다. 완료되면 Drive에 저장되어 재실행 시 다시 계산하지 않습니다.

In [ ]:
from sentence_transformers import SentenceTransformer
cache = WORK/'embeddings_31.npz'
if cache.exists():
    z=np.load(cache,allow_pickle=True)
    query_emb, product_emb, ruling_emb = z['query_emb'], z['product_emb'], z['ruling_emb']
    print('Drive 캐시에서 임베딩 불러옴')
else:
    sbert=SentenceTransformer('intfloat/multilingual-e5-base')
    query_emb=sbert.encode(['query: '+str(x) for x in eval_df.query_text],batch_size=16,show_progress_bar=True,normalize_embeddings=True,convert_to_numpy=True)
    product_emb=sbert.encode(['passage: '+str(x) for x in eval_df.query_text],batch_size=16,show_progress_bar=True,normalize_embeddings=True,convert_to_numpy=True)
    ruling_emb=sbert.encode(['passage: '+str(x) for x in eval_df.ruling_text],batch_size=16,show_progress_bar=True,normalize_embeddings=True,convert_to_numpy=True)
    np.savez_compressed(cache,query_emb=query_emb,product_emb=product_emb,ruling_emb=ruling_emb)
    print('임베딩 캐시 저장 완료')
print(query_emb.shape,product_emb.shape,ruling_emb.shape)

## 3. Cross-Encoder 준비

다국어 문장쌍 재순위화 모델을 불러옵니다. 모델 다운로드가 끝나면 다음 셀로 이동합니다.

In [ ]:
CE_NAME='cross-encoder/mmarco-mMiniLMv2-L12-H384-v1'
cross_encoder=CrossEncoder(CE_NAME,max_length=384)
print('Cross-Encoder 준비 완료:',CE_NAME)

## 4. Fold별 평가 함수 준비

각 평가 제품군은 학습 사례와 후보 검색에서 제외합니다. Fold마다 결과를 Drive에 저장합니다.

In [ ]:
def rank_case_codes(scores,codes):
    return pd.DataFrame({'hs6':np.asarray(codes).astype(str),'score':np.asarray(scores,dtype=float)}).groupby('hs6').score.max().sort_values(ascending=False)
def rrf(a,b,k=60,bw=0.35):
    ap={str(c):i+1 for i,c in enumerate(a.index)}
    bp={str(c):i+1 for i,c in enumerate(b.index)}
    return pd.Series({c:1/(k+ap.get(c,99999))+bw/(k+bp.get(c,99999)) for c in labels}).sort_values(ascending=False)
def norm(x):
    x=np.asarray(x,dtype=float); lo,hi=x.min(),x.max()
    return (x-lo)/(hi-lo) if hi>lo else np.ones(len(x))
def code_rank(scores,codes):
    d=pd.DataFrame({'hs6':np.asarray(codes).astype(str),'score':np.asarray(scores,dtype=float)})
    s=d.groupby('hs6').score.apply(lambda x:x.nlargest(min(3,len(x))).mean())
    return s.reindex(labels,fill_value=-1).sort_values(ascending=False)
def get_candidates(case_scores,struct_scores,train_codes,limit=35):
    ix=set(np.argsort(-case_scores)[:25].tolist())|set(np.argsort(-struct_scores)[:25].tolist())
    return sorted(ix,key=lambda j:0.65*case_scores[j]+0.35*struct_scores[j],reverse=True)[:limit]
def result_path(fold):
    return WORK/f'fold_{fold:02d}.csv'
print('평가 함수 준비 완료')

## 5. Fold 실행

한 번에 전체 실행하지 말고, 아래 셀에서 `FOLD_START`와 `FOLD_END`를 지정해 3~5개 Fold씩 실행하세요. 이미 저장된 Fold는 자동으로 건너뜁니다.

In [ ]:
FOLD_START=1
FOLD_END=5
splits=list(LeaveOneGroupOut().split(eval_df,eval_df.hs6,groups=eval_df.product_family))
for fold,(tr,te) in enumerate(splits,1):
    if not FOLD_START<=fold<=FOLD_END: continue
    if result_path(fold).exists():
        print('Fold',fold,'이미 완료 - 건너뜀'); continue
    train=eval_df.iloc[tr].copy(); train_codes=train.hs6.astype(str).to_numpy()
    struct_train=(train.form_focus_text.astype(str)+' '+train.discriminative_text.astype(str)).tolist()
    struct_test=(eval_df.iloc[te].form_focus_text.astype(str)+' '+eval_df.iloc[te].discriminative_text.astype(str)).tolist()
    vec=TfidfVectorizer(analyzer='char_wb',ngram_range=(2,5),min_df=1,max_features=20000,sublinear_tf=True)
    xt=vec.fit_transform(struct_train); xv=vec.transform(struct_test)
    rows=[]
    for pos,global_i in enumerate(te):
        case_scores=0.5*(product_emb[tr]@query_emb[global_i])+0.5*(ruling_emb[tr]@query_emb[global_i])
        struct_scores=cosine_similarity(xv[pos],xt).ravel()
        base_rank=rank_case_codes(case_scores,train_codes)
        struct_rank=rank_case_codes(struct_scores,train_codes)
        hybrid_rank=rrf(base_rank,struct_rank)
        candidate_ids=get_candidates(case_scores,struct_scores,train_codes,35)
        pairs=[(str(eval_df.iloc[global_i].query_text),str(train.iloc[j].ruling_text)) for j in candidate_ids]
        ce_scores=np.asarray(cross_encoder.predict(pairs,batch_size=8,show_progress_bar=False),dtype=float)
        ce_rank=code_rank(ce_scores,train_codes[candidate_ids])
        hybrid_case=0.65*norm(ce_scores)+0.35*norm(np.asarray([case_scores[j] for j in candidate_ids]))
        hybrid_ce_rank=code_rank(hybrid_case,train_codes[candidate_ids])
        options={'SBERT_product_plus_ruling':base_rank,'Hybrid_Structure_RRF':hybrid_rank,'CrossEncoder_Reranking':ce_rank,'Hybrid_CrossEncoder':hybrid_ce_rank}
        for model,rank in options.items():
            rows.append({'row_index':int(global_i),'fold':fold,'heldout_family':eval_df.iloc[global_i].product_family,'true_hs6':str(eval_df.iloc[global_i].hs6),'true_hs4':str(eval_df.iloc[global_i].hs4),'model':model,'predicted_hs6':str(rank.index[0]),'top3_candidates':'|'.join(map(str,rank.index[:3]))})
    pd.DataFrame(rows).to_csv(result_path(fold),index=False,encoding='utf-8-sig')
    print(f'Fold {fold}/{len(splits)} 저장 완료 - 평가 사례 {len(te)}건')
print('지정한 Fold 실행 완료')

## 6. 실행 진행률 확인

각 Fold가 끝날 때마다 CSV가 Drive에 저장됩니다. 런타임이 끊기면 0번부터 다시 연결하고 1~4번 셀을 실행한 뒤, 미완료 Fold 구간을 지정하면 됩니다.

In [ ]:
done=sorted(int(p.stem.split('_')[1]) for p in WORK.glob('fold_*.csv'))
print('완료 Fold:',done)
print('전체 Fold:',len(splits),'| 남은 Fold:',len(splits)-len(done))

## 7. 전체 결과 통합 및 평가

모든 Fold가 완료된 후 실행합니다.

In [ ]:
files=sorted(WORK.glob('fold_*.csv'))
if len(files)!=len(splits): raise ValueError(f'아직 {len(splits)-len(files)}개 Fold가 남았습니다.')
pred=pd.concat([pd.read_csv(p) for p in files],ignore_index=True)
pred['top3_list']=pred.top3_candidates.fillna('').str.split('|')
def metrics(g):
    yt=g.true_hs6.astype(str); yp=g.predicted_hs6.astype(str)
    return pd.Series({'n':len(g),'hs4_accuracy':np.mean(yt.str[:4].to_numpy()==yp.str[:4].to_numpy()),'hs6_accuracy':accuracy_score(yt,yp),'macro_f1_hs6':f1_score(yt,yp,labels=labels,average='macro',zero_division=0),'weighted_f1_hs6':f1_score(yt,yp,average='weighted',zero_division=0),'top3_hs6_accuracy':np.mean([t in c for t,c in zip(yt,g.top3_list)])})
summary=pred.groupby('model').apply(metrics,include_groups=False).reset_index().sort_values('macro_f1_hs6',ascending=False)
class_rows=[]
for model,g in pred.groupby('model'):
    p,r,f,s=precision_recall_fscore_support(g.true_hs6.astype(str),g.predicted_hs6.astype(str),labels=labels,zero_division=0)
    class_rows.extend([{'model':model,'hs6':c,'support':int(n),'precision':a,'recall':b,'f1':d} for c,a,b,d,n in zip(labels,p,r,f,s)])
class_metrics=pd.DataFrame(class_rows)
display(summary.round(4)); display(class_metrics.sort_values(['model','f1']).round(4))
pred.to_csv(WORK/'31_oof_predictions.csv',index=False,encoding='utf-8-sig')
summary.to_csv(WORK/'31_model_comparison.csv',index=False,encoding='utf-8-sig')
class_metrics.to_csv(WORK/'31_class_metrics.csv',index=False,encoding='utf-8-sig')
with zipfile.ZipFile(WORK/'31_hybrid_crossencoder_results.zip','w',zipfile.ZIP_DEFLATED) as z:
    for p in WORK.glob('31_*.csv'): z.write(p,p.name)
print('최종 결과:',WORK/'31_hybrid_crossencoder_results.zip')